In [12]:
!pip install -q sentence-transformers faiss-cpu pdfplumber transformers gradio


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
import pdfplumber
import os

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

# Load multiple PDFs from a directory
pdf_dir = "/kaggle/input/hr-data/"
all_text = ""
for filename in os.listdir(pdf_dir):
    if filename.endswith(".pdf"):
        pdf_path = os.path.join(pdf_dir, filename)
        all_text += extract_text_from_pdf(pdf_path)


In [14]:
def chunk_text(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = chunk_text(all_text)


In [15]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedding_model.encode(chunks)
embedding_matrix = np.array(embeddings).astype('float32')


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
import faiss
import os

index_file = "faiss_index.index"

if os.path.exists(index_file):
    faiss_index = faiss.read_index(index_file)
else:
    faiss_index = faiss.IndexFlatL2(embedding_matrix.shape[1])
    faiss_index.add(embedding_matrix)
    faiss.write_index(faiss_index, index_file)


In [17]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")


Device set to use cpu


In [18]:
chat_history = []

def answer_query(query):
    top_k=5
    query_embedding = embedding_model.encode([query]).astype("float32")
    distances, indices = faiss_index.search(query_embedding, top_k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = " ".join(retrieved_chunks)
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    response = generator(prompt, max_length=150, do_sample=False)
    answer = str(response[0]['generated_text'])
    chat_history.append((query, answer))
    return answer


In [19]:
!pip install portkey_ai langchain_openai

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
import os
from portkey_ai import createHeaders, PORTKEY_GATEWAY_URL
from langchain_openai import ChatOpenAI

# FAISS-based chat history setup
chat_history = []

# Directly use your test API key (⚠️ for testing only)
PORTKEY_API_KEY = "Oa1D+Q+wZ7BTNO9OMWK/g6y1jhFI"
portkey_headers = createHeaders(api_key=PORTKEY_API_KEY)

# Initialize Portkey LLM
portkey_llm = ChatOpenAI(
    api_key=PORTKEY_API_KEY,
    base_url=PORTKEY_GATEWAY_URL,
    default_headers=portkey_headers
)

def answer_query(query):
    top_k = 5
    query_embedding = embedding_model.encode([query]).astype("float32")
    distances, indices = faiss_index.search(query_embedding, top_k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = " ".join(retrieved_chunks)
    
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    
    # Use Portkey LLM to generate the response
    response = portkey_llm.invoke(prompt)
    answer = str(response.content)
    
    chat_history.append((query, answer))
    return answer

# Simple interactive chat loop
print("Welcome to the QnA Chatbot (Portkey Test)! Type 'exit' to end the chat.\n")

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "exit":
        print("Goodbye!")
        break
    
    response = answer_query(user_input)
    print(f"Bot: {response}\n")


Welcome to the QnA Chatbot (Portkey Test)! Type 'exit' to end the chat.



You:  what are Anti-Harassment Policy?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Bot: Based on the provided context:

The Anti-Harassment Policy states that Innovexa Technologies is committed to maintaining a workplace free of harassment. It also specifies that any misconduct should be reported immediately.

